In [1]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import datetime as dt
import sys

In [2]:
# Pre-processing

# Put all data together
df = pd.concat([
    pd.read_csv("../data/processed/sentiment/fmb_standardized.csv"),
    pd.read_csv("../data/processed/sentiment/fpb_standardized.csv"), 
    pd.read_csv("../data/processed/sentiment/stitched_sentiment.csv"),
], ignore_index=True)

df['sentiment'] = df['sentiment'].astype(str).str.lower().str.strip()
df = df.dropna(subset=['sentiment'])

# Split data
train_val, test = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['sentiment'],
)

train, val = train_test_split(
    train_val, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_val['sentiment']
)
# percentage split of (train, val, test) = (.64, .16, .20)
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 16605, Val: 4152, Test: 5190


## Loading [L&M Dictionary](https://sraf.nd.edu/loughranmcdonald-master-dictionary/)

### 7 Sentiment Categories
1. negative - words that signal bad news, weakness, risk, deterioration, loss, failure, or adverse outcomes (e.g.: impairment, risk, deficit)

2. positive - words that signal good news, strength, improvement, gain, success, or favorable outcomes (e.g.: improve, benefit, strong, achieve, growth) +ve

3. uncertainty - words that show doubt, ambiguity, unpredictability, or lack of confidence about what may happen (e.g.: uncertain, approximately, may, possible, pending, volatile) neutral

4. litigious - words related to legal disputes, lawsuits, regulation, claims, contracts, investigations, or legal liability (e.g.: litigation, plaintiff, settlement, claim, court, investigation) -ve

5. strong modal - words that express strong obligation, certainty, or firm intention (must, will, shall) neutral

6. weak modal - words that express possibility, flexibility, or weaker commitment (e.g.: may, might, could, depending) neutral

7. constraining - words that indicate limitations, restrictions, constraints, pressure, or barriers on actions or outcomes (e.g.: restrict, limit, requirement, bound, constrained) -ve

In [3]:
def load_masterdictionary(file_path, print_flag=False, f_log=None, get_other=False):
    start_local = dt.datetime.now()
    # Setup dictionaries
    _master_dictionary = {}
    _sentiment_categories = ['negative', 'positive', 'uncertainty', 'litigious', 
                             'strong_modal', 'weak_modal', 'constraining', 'complexity']
    _sentiment_dictionaries = dict()
    for sentiment in _sentiment_categories:
        _sentiment_dictionaries[sentiment] = dict()
   
    # Load slightly modified common stopwords. 
    # Dropped from traditional: A, I, S, T, DON, WILL, AGAINST
    # Added: AMONG
    _stopwords = ['ME', 'MY', 'MYSELF', 'WE', 'OUR', 'OURS', 'OURSELVES', 'YOU', 'YOUR', 'YOURS',
                  'YOURSELF', 'YOURSELVES', 'HE', 'HIM', 'HIS', 'HIMSELF', 'SHE', 'HER', 'HERS', 'HERSELF',
                  'IT', 'ITS', 'ITSELF', 'THEY', 'THEM', 'THEIR', 'THEIRS', 'THEMSELVES', 'WHAT', 'WHICH',
                  'WHO', 'WHOM', 'THIS', 'THAT', 'THESE', 'THOSE', 'AM', 'IS', 'ARE', 'WAS', 'WERE', 'BE',
                  'BEEN', 'BEING', 'HAVE', 'HAS', 'HAD', 'HAVING', 'DO', 'DOES', 'DID', 'DOING', 'AN',
                  'THE', 'AND', 'BUT', 'IF', 'OR', 'BECAUSE', 'AS', 'UNTIL', 'WHILE', 'OF', 'AT', 'BY',
                  'FOR', 'WITH', 'ABOUT', 'BETWEEN', 'INTO', 'THROUGH', 'DURING', 'BEFORE',
                  'AFTER', 'ABOVE', 'BELOW', 'TO', 'FROM', 'UP', 'DOWN', 'IN', 'OUT', 'ON', 'OFF', 'OVER',
                  'UNDER', 'AGAIN', 'FURTHER', 'THEN', 'ONCE', 'HERE', 'THERE', 'WHEN', 'WHERE', 'WHY',
                  'HOW', 'ALL', 'ANY', 'BOTH', 'EACH', 'FEW', 'MORE', 'MOST', 'OTHER', 'SOME', 'SUCH',
                  'NO', 'NOR', 'NOT', 'ONLY', 'OWN', 'SAME', 'SO', 'THAN', 'TOO', 'VERY', 'CAN',
                  'JUST', 'SHOULD', 'NOW', 'AMONG']

    # Loop thru words and load dictionaries
    with open(file_path) as f:
        _total_documents = 0
        _md_header = f.readline()  # Consume header line

        for line in f:
            cols = line.rstrip('\n').split(',')
            word = cols[0]
            _master_dictionary[word] = MasterDictionary(cols, _stopwords)
            for sentiment in _sentiment_categories:
                if getattr(_master_dictionary[word], sentiment):
                    _sentiment_dictionaries[sentiment][word] = 0
            _total_documents += _master_dictionary[cols[0]].doc_count
            if len(_master_dictionary) % 5000 == 0 and print_flag:
                print(f'\r ...Loading Master Dictionary {len(_master_dictionary):,}', end='', flush=True)

    if print_flag:
        print('\r', end='')  # clear line
        print(f'\nMaster Dictionary loaded from file:\n  {file_path}\n')
        print(f'  master_dictionary has {len(_master_dictionary):,} words.\n')

    if f_log:
        try:
            f_log.write('\n\n  FUNCTION: load_masterdictionary' +
                        '(file_path, print_flag, f_log, get_other)\n')
            f_log.write(f'\n    file_path  = {file_path}')
            f_log.write(f'\n    print_flag = {print_flag}')
            f_log.write(f'\n    f_log      = {f_log.name}')
            f_log.write(f'\n    get_other  = {get_other}')
            f_log.write(f'\n\n    {len(_master_dictionary):,} words loaded in master_dictionary.\n')
            f_log.write(f'\n    Sentiment:')
            for sentiment in _sentiment_categories:
                f_log.write(f'\n      {sentiment:13}: {len(_sentiment_dictionaries[sentiment]):8,}')
            f_log.write(f'\n\n  END FUNCTION: load_masterdictionary: {(dt.datetime.now()-start_local)}')
        except Exception as e:
            print('Log file in load_masterdictionary is not available for writing')
            print(f'Error = {e}')

    if get_other:
        return _master_dictionary, _md_header, _sentiment_categories, _sentiment_dictionaries, _stopwords, _total_documents
    else:
        return _master_dictionary


class MasterDictionary:
    def __init__(self, cols, _stopwords):
        for ptr, col in enumerate(cols):
            if col == '':
                cols[ptr] = '0'
        try:
            self.word = cols[0].upper()
            self.sequence_number = int(cols[1])    
            self.word_count = int(cols[2])
            self.word_proportion = float(cols[3])
            self.average_proportion = float(cols[4])
            self.std_dev_prop = float(cols[5])
            self.doc_count = int(cols[6])
            self.negative = int(cols[7])
            self.positive = int(cols[8])
            self.uncertainty = int(cols[9])
            self.litigious = int(cols[10])
            self.strong_modal = int(cols[11])
            self.weak_modal = int(cols[12])
            self.constraining = int(cols[13])
            self.complexity = int(cols[14])
            self.syllables = int(cols[15])
            self.source = cols[16]
            if self.word in _stopwords:
                self.stopword = True
            else:
                self.stopword = False
        except:
            print('ERROR in class MasterDictionary')
            print(f'word = {cols[0]} : seqnum = {cols[1]}')
            quit()
        return

In [4]:
start = dt.datetime.now()
print(f'\n\n{start.strftime("%c")}\nPROGRAM NAME: {sys.argv[0]}\n')
f_log = open('./Load_MD_Logfile.txt', 'w')
md = (r'../data/Loughran-McDonald_MasterDictionary_1993-2025.csv')
master_dictionary, md_header, sentiment_categories, sentiment_dictionaries, stopwords, total_documents = \
    load_masterdictionary(md, True, f_log, True)
print(f'\n\nRuntime: {(dt.datetime.now()-start)}')
print(f'\nNormal termination.\n{dt.datetime.now().strftime("%c")}\n')



Thu Apr 16 20:37:12 2026
PROGRAM NAME: /home/omar02/comp6713/comp6713-veil/.venv/lib/python3.10/site-packages/ipykernel_launcher.py

 ...Loading Master Dictionary 85,000
Master Dictionary loaded from file:
  ../data/Loughran-McDonald_MasterDictionary_1993-2025.csv

  master_dictionary has 86,553 words.



Runtime: 0:00:00.868264

Normal termination.
Thu Apr 16 20:37:13 2026



In [7]:
print(master_dictionary['AARDVARK'].weak)

AttributeError: 'MasterDictionary' object has no attribute 'weak'

### Simple rule based model 
Assign sentiment based on majority sentiment token
- Positive: [postive]
- Negative: [negative, litiguous, constraining]
- Neutral: [uncertainty, strong_modal, weak_modal, cns]

In [11]:
import re

# TODO map_function : string -> [(positive 
#                   | negative 
#                   | uncertainty 
#                   | litiguous 
#                   | strong_modal 
#                   | weak_modal 
#                   | constraining 
#                   | cns (Can Not Say) )]
# Map each token with its corresponding sentiment
def map_function(text):
    text = re.split(r"[,:'\s.]", text)
    text = [text for text in text if len(text) > 1]
    text_sentiment = []
    for string in text:
        string = string.upper()
        try:
            string_md = master_dictionary[string]
            if (string_md.stopword): continue
            if string_md.positive > 0: text_sentiment.append("positive")
            elif string_md.negative > 0: text_sentiment.append("negative")
            elif string_md.uncertainty > 0: text_sentiment.append("uncertainty")
            elif string_md.litigious > 0: text_sentiment.append("litigious")
            elif string_md.strong_modal > 0: text_sentiment.append("strong_modal")
            elif string_md.weak_modal > 0: text_sentiment.append("weak_modal")
            elif string_md.constraining > 0: text_sentiment.append("constraining")
            else: text_sentiment.append("cns")
        except KeyError: 
            text_sentiment.append("cns")
    return text_sentiment

In [12]:
# apply to each row of the data column
df_new = df["data"].apply(map_function)

In [14]:
print(df_new)

0                 [cns, cns, cns, cns, cns, cns, cns, cns]
1                      [cns, cns, cns, cns, cns, positive]
2                 [cns, cns, cns, cns, cns, cns, cns, cns]
3                 [cns, cns, cns, cns, cns, cns, cns, cns]
4        [cns, cns, cns, cns, cns, cns, cns, cns, posit...
                               ...                        
25942    [cns, cns, cns, cns, cns, cns, cns, cns, cns, ...
25943    [cns, cns, cns, cns, cns, cns, cns, cns, cns, ...
25944    [cns, cns, cns, cns, cns, cns, cns, cns, cns, ...
25945    [cns, cns, cns, cns, cns, cns, cns, cns, cns, ...
25946    [cns, cns, cns, cns, cns, cns, cns, cns, negat...
Name: data, Length: 25947, dtype: object
